In [0]:
from pyspark.sql import functions as F

# -----------------------------
# 1. Load tables
# -----------------------------
bronze_jobs = spark.table("bronze_jobs")
bronze_candidates = spark.table("bronze_candidates")
bronze_education = spark.table("bronze_education")
bronze_applications = spark.table("bronze_applications")
bronze_workflow_events = spark.table("bronze_workflow_events")

dim_job = spark.table("dim_job")
dim_candidate = spark.table("dim_candidate")
fct_workflow_events = spark.table("fct_workflow_events")
fct_applications = spark.table("fct_applications")

# -----------------------------
# 2. Helper to standardize check output
# -----------------------------
def build_check_result(check_name, failed_count, table_name):
    status = "PASS" if failed_count == 0 else "FAIL"
    return [(check_name, table_name, failed_count, status)]

# -----------------------------
# 3. Duplicate checks
# -----------------------------
dup_jobs = bronze_jobs.groupBy("job_id").count().filter(F.col("count") > 1).count()
dup_candidates = bronze_candidates.groupBy("candidate_id").count().filter(F.col("count") > 1).count()
dup_applications = bronze_applications.groupBy("application_id").count().filter(F.col("count") > 1).count()
dup_events = bronze_workflow_events.groupBy("event_id").count().filter(F.col("count") > 1).count()

dup_dim_job = dim_job.groupBy("job_id").count().filter(F.col("count") > 1).count()
dup_dim_candidate = dim_candidate.groupBy("candidate_id").count().filter(F.col("count") > 1).count()
dup_fct_applications = fct_applications.groupBy("application_id").count().filter(F.col("count") > 1).count()

# -----------------------------
# 4. Null checks
# -----------------------------
null_job_id = bronze_jobs.filter(F.col("job_id").isNull()).count()
null_candidate_id = bronze_candidates.filter(F.col("candidate_id").isNull()).count()
null_application_id = bronze_applications.filter(F.col("application_id").isNull()).count()
null_apply_date = bronze_applications.filter(F.col("apply_date").isNull()).count()
null_event_timestamp = bronze_workflow_events.filter(F.col("event_timestamp").isNull()).count()

# -----------------------------
# 5. Volume checks
# -----------------------------
row_count_jobs = bronze_jobs.count()
row_count_candidates = bronze_candidates.count()
row_count_education = bronze_education.count()
row_count_applications = bronze_applications.count()
row_count_events = bronze_workflow_events.count()

volume_results = [
    ("row_count_bronze_jobs", "bronze_jobs", row_count_jobs, "INFO"),
    ("row_count_bronze_candidates", "bronze_candidates", row_count_candidates, "INFO"),
    ("row_count_bronze_education", "bronze_education", row_count_education, "INFO"),
    ("row_count_bronze_applications", "bronze_applications", row_count_applications, "INFO"),
    ("row_count_bronze_workflow_events", "bronze_workflow_events", row_count_events, "INFO"),
]

# -----------------------------
# 6. Anomaly detection: hired before applied
# -----------------------------
anomaly_hired_before_applied = (
    fct_applications
    .withColumn("apply_date", F.to_date("apply_date"))
    .withColumn("hired_date", F.to_date("hired_date"))
    .filter(
        F.col("hired_date").isNotNull() &
        F.col("apply_date").isNotNull() &
        (F.col("hired_date") < F.col("apply_date"))
    )
    .select(
        "application_id",
        "job_id",
        "candidate_id",
        "apply_date",
        "hired_date"
    )
)

anomaly_count = anomaly_hired_before_applied.count()

# Save anomaly table
anomaly_hired_before_applied.write.format("delta").mode("overwrite").saveAsTable("dq_hired_before_applied")

# -----------------------------
# 7. Assemble quality results
# -----------------------------
results = []
results += build_check_result("duplicate_job_id_bronze", dup_jobs, "bronze_jobs")
results += build_check_result("duplicate_candidate_id_bronze", dup_candidates, "bronze_candidates")
results += build_check_result("duplicate_application_id_bronze", dup_applications, "bronze_applications")
results += build_check_result("duplicate_event_id_bronze", dup_events, "bronze_workflow_events")

results += build_check_result("duplicate_job_id_dim", dup_dim_job, "dim_job")
results += build_check_result("duplicate_candidate_id_dim", dup_dim_candidate, "dim_candidate")
results += build_check_result("duplicate_application_id_fact", dup_fct_applications, "fct_applications")

results += build_check_result("null_job_id", null_job_id, "bronze_jobs")
results += build_check_result("null_candidate_id", null_candidate_id, "bronze_candidates")
results += build_check_result("null_application_id", null_application_id, "bronze_applications")
results += build_check_result("null_apply_date", null_apply_date, "bronze_applications")
results += build_check_result("null_event_timestamp", null_event_timestamp, "bronze_workflow_events")

results += build_check_result("hired_before_applied", anomaly_count, "fct_applications")
results += volume_results

dq_results = spark.createDataFrame(
    results,
    schema=["check_name", "table_name", "failed_or_observed_count", "status"]
)

# -----------------------------
# 8. Save and display
# -----------------------------
dq_results.write.format("delta").mode("overwrite").saveAsTable("dq_results")

display(dq_results.orderBy("status", "check_name"))
display(anomaly_hired_before_applied)

In [0]:
spark.sql("SELECT * FROM dq_results ORDER BY status, check_name").show(truncate=False)
spark.sql("SELECT * FROM dq_hired_before_applied LIMIT 20").show()